In [ ]:
import json\nfrom difflib import SequenceMatcher\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport pandas as pd\nimport spacy\nfrom datasets import load_dataset\nfrom sklearn.model_selection import train_test_split\n\nfrom src.utils.config import load_config\n\nconfig = load_config()\nprocessed_dir = Path(config.data.processed_data_path)\nprocessed_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
raw_dataset = load_dataset(config.data.jfleg_name)\nsplit_frames = {name: split.to_pandas() for name, split in raw_dataset.items()}\nshape_summary = {name: frame.shape for name, frame in split_frames.items()}\nall_examples = pd.concat(split_frames.values(), ignore_index=True)\n\nprint('Split shapes:', shape_summary)\nprint('Columns:', list(all_examples.columns))\nall_examples.head()

In [ ]:
preview = all_examples[['sentence', 'corrections']].head(5).copy()\npreview['corrected'] = preview['corrections'].apply(lambda items: items[0] if items else '')\npreview[['sentence', 'corrected']]

In [ ]:
sentence_lengths = all_examples['sentence'].str.split().str.len()\nplt.figure(figsize=(10, 5))\nplt.hist(sentence_lengths, bins=25, color='#2E86AB', edgecolor='white')\nplt.title('JFLEG Sentence Length Distribution')\nplt.xlabel('Tokens per sentence')\nplt.ylabel('Count')\nplt.show()

In [ ]:
shuffled_examples = all_examples.sample(frac=1.0, random_state=42).reset_index(drop=True)\ntrain_df, temp_df = train_test_split(\n    shuffled_examples,\n    train_size=config.data.train_split,\n    random_state=42,\n)\nvalidation_fraction = config.data.val_split / (config.data.val_split + config.data.test_split)\nval_df, test_df = train_test_split(\n    temp_df,\n    train_size=validation_fraction,\n    random_state=42,\n)\n\ntrain_tokens = [token.lower() for sentence in train_df['sentence'] for token in sentence.split()]\nval_test_tokens = [token.lower() for sentence in pd.concat([val_df['sentence'], test_df['sentence']]) for token in sentence.split()]\ntrain_vocab = set(train_tokens)\nunique_tokens = set(token.lower() for sentence in all_examples['sentence'] for token in sentence.split())\navg_sentence_length = sentence_lengths.mean()\noov_tokens = [token for token in val_test_tokens if token not in train_vocab]\noov_rate = len(oov_tokens) / max(len(val_test_tokens), 1)\n\nprint('Unique tokens:', len(unique_tokens))\nprint('Average sentence length:', round(avg_sentence_length, 2))\nprint('OOV rate:', round(oov_rate, 4))

In [ ]:
try:\n    nlp = spacy.load('en_core_web_sm')\nexcept OSError:\n    nlp = spacy.blank('en')\n\ndef classify_error_type(original: str, corrected: str) -> str:\n    punctuation_chars = set('.,!?;:\'\"')\n    if any(char in punctuation_chars for char in set(original) ^ set(corrected)):\n        return 'punctuation'\n\n    similarity = SequenceMatcher(None, original.lower(), corrected.lower()).ratio()\n    if similarity > 0.88:\n        return 'spelling'\n\n    original_doc = nlp(original)\n    corrected_doc = nlp(corrected)\n    if len(original_doc) > 0 and len(corrected_doc) > 0:\n        paired_tags = zip([token.pos_ for token in original_doc], [token.pos_ for token in corrected_doc])\n        if any(left != right for left, right in paired_tags):\n            return 'grammar'\n\n    return 'grammar'\n\nall_examples['corrected'] = all_examples['corrections'].apply(lambda items: items[0] if items else '')\nall_examples['error_type'] = all_examples.apply(\n    lambda row: classify_error_type(row['sentence'], row['corrected']),\n    axis=1,\n)\nall_examples['error_type'].value_counts()

In [ ]:
error_samples = (\n    all_examples[['sentence', 'corrected', 'error_type']]\n    .groupby('error_type', group_keys=False)\n    .head(3)\n    .reset_index(drop=True)\n)\nerror_samples

In [ ]:
split_stats = pd.DataFrame(\n    [\n        {'split': 'train', 'rows': len(train_df), 'ratio': len(train_df) / len(all_examples)},\n        {'split': 'validation', 'rows': len(val_df), 'ratio': len(val_df) / len(all_examples)},\n        {'split': 'test', 'rows': len(test_df), 'ratio': len(test_df) / len(all_examples)},\n    ]\n)\nsplit_stats

In [ ]:
for split_name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:\n    records = frame[['sentence', 'corrected', 'error_type']].rename(columns={'sentence': 'original'}).to_dict(orient='records')\n    output_path = processed_dir / f'jfleg_{split_name}.jsonl'\n    with output_path.open('w', encoding='utf-8') as output_file:\n        for record in records:\n            output_file.write(json.dumps(record, ensure_ascii=False) + '\\n')\n    print(f'Saved {split_name} split to {output_path}')

# Summary\n\n- Loaded the public JFLEG grammar dataset through the Hugging Face `datasets` library.\n- Reviewed example correction pairs and sentence-length distribution.\n- Calculated vocabulary size, average length, and an approximate OOV rate based on a train/validation/test split.\n- Labeled examples with simple spelling, grammar, and punctuation heuristics for exploratory analysis.\n- Saved processed JSON Lines files to `data/processed/` for later training and evaluation sprints.